# 09 — Benchmark and placebo checks

`05` fits the core asymmetry model on WTI crude and national gasoline retail. This notebook adds two checks: Brent as a robustness check on the crude benchmark, and diesel as a comparison series to assess whether the pattern is specific to gasoline or broader. Both roles are recorded in `src/series_manifest.py`.

In [1]:
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("no pyproject.toml found in any parent directory")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.asymmetry import build_design_matrix, fit_distributed_lag, test_asymmetry
from src.lag_structure import cross_correlation
from scripts.verify_alignment import PROCESSED_DIR, load_series

K = 4  # same primary spec as 05_asymmetry.ipynb

retail_gas = load_series(PROCESSED_DIR / "retail.csv", "GASREGW", "weekly-mon")
retail_diesel = load_series(PROCESSED_DIR / "retail.csv", "GASDESW", "weekly-mon")
crude_wti = load_series(PROCESSED_DIR / "crude.csv", "WCOILWTICO", "weekly-fri")
crude_brent = load_series(PROCESSED_DIR / "crude.csv", "WCOILBRENTEU", "weekly-fri")

# headline numbers from 05_asymmetry.ipynb §5, primary spec (WTI crude -> national gasoline
# retail, K=4) — quoted here as a fixed reference, not recomputed
headline = pd.DataFrame(
    [
        {"horizon": 0, "estimate": 0.4333, "p_value": 0.0001},
        {"horizon": 1, "estimate": 0.2331, "p_value": 0.0239},
        {"horizon": 2, "estimate": 0.2055, "p_value": 0.0823},
        {"horizon": 3, "estimate": 0.1169, "p_value": 0.3570},
        {"horizon": 4, "estimate": 0.0303, "p_value": 0.8137},
    ]
).set_index("horizon")

## 1. Brent instead of WTI

Same spec as `05`: crude -> national gasoline retail, K = 4. Only the crude series changes, WTI
to Brent (`WCOILBRENTEU`). The cross-correlation below is a quick check that K = 4 isn't obviously
the wrong lag for Brent before trusting a fit built on it, not a full re-derivation like `04`.
`04`'s own WTI lag-0 correlation was 0.592, decaying sharply by lag 2. A similar shape here means
K = 4 still applies; a very different one would mean this comparison needs its own K first.

In [2]:
brent_corr = cross_correlation(retail_gas, crude_brent, max_lag=6, mode="weekly")
brent_corr.round(3)

,lag,correlation,n_obs
0,0,0.631,865
1,1,0.375,864
2,2,0.179,863
3,3,0.151,862
4,4,0.130,861
5,5,0.057,860
6,6,0.092,859


In [3]:
X_brent = build_design_matrix(retail_gas, crude_brent, K=K, mode="weekly")
res_brent = fit_distributed_lag(X_brent)

brent_result = pd.DataFrame(
    [test_asymmetry(res_brent, K=K, horizon=h) for h in (0, 1, K)]
).set_index("horizon")[["estimate", "p_value"]]

pd.concat({"WTI (05, headline)": headline, "Brent": brent_result}, axis=1).round(4)

build_design_matrix: dropped 5 of 866 rows to NaN (differencing + 4 lag(s)); 861 rows remain


WTI (05, headline)            Brent        
                  estimate p_value estimate p_value
horizon                                            
0                   0.4333  0.0001   0.3692  0.0011
1                   0.2331  0.0239   0.2448  0.0529
2                   0.2055  0.0823      NaN     NaN
3                   0.1169  0.3570      NaN     NaN
4                   0.0303  0.8137   0.0499  0.7194

Brent gives a similar overall pattern to WTI. The same-week gap is significant under Brent too (0.37, p = 0.001), somewhat smaller than WTI's 0.43. At h = 1, however, the Brent estimate is only marginally significant (0.24, p = 0.053), consistent with h = 1 being the least stable horizon in `05`. By h = 4, the gap is indistinguishable from zero under both benchmarks.

This completes the planned robustness check using Brent as an alternative crude benchmark.

## 2. Diesel instead of gasoline

Same specification crude (WTI) -> retail, with national diesel (`GASDESW`) as the retail series
instead of gasoline. Diesel is refined from the same crude but sold through a separate retail
market from gasoline, which makes it a check on whether the pattern found in `05` is specific to
gasoline's retail market or broader.

Unlike Brent, the lag structure for diesel does not support using K = 4. Diesel's own cross-correlation below decays faster than
gasoline's: lag 3 (0.089) is still just above the ~0.067 noise band, but lag 4 (0.055) has already
dropped inside it. The correlation has fallen into the noise band by lag 4, so K = 3 is used for the diesel specification.

In [4]:
diesel_corr = cross_correlation(retail_diesel, crude_wti, max_lag=6, mode="weekly")
diesel_corr.round(3)

,lag,correlation,n_obs
0,0,0.577,865
1,1,0.305,864
2,2,0.155,863
3,3,0.089,862
4,4,0.055,861
5,5,0.060,860
6,6,0.038,859


In [5]:
K_diesel = 3  # diesel's own cross-correlation puts the signal-to-noise cutoff at lag 3, not 4

X_diesel = build_design_matrix(retail_diesel, crude_wti, K=K_diesel, mode="weekly")
res_diesel = fit_distributed_lag(X_diesel)

diesel_result = pd.DataFrame(
    [test_asymmetry(res_diesel, K=K_diesel, horizon=h) for h in (0, 1, K_diesel)]
).set_index("horizon")[["estimate", "p_value"]]

pd.concat({"gasoline (05, headline)": headline, "diesel (K=3)": diesel_result}, axis=1).round(4)

build_design_matrix: dropped 4 of 866 rows to NaN (differencing + 3 lag(s)); 862 rows remain


gasoline (05, headline)         diesel (K=3)        
                       estimate p_value     estimate p_value
horizon                                                     
0                        0.4333  0.0001       0.6889  0.0022
1                        0.2331  0.0239       0.5002  0.0172
2                        0.2055  0.0823          NaN     NaN
3                        0.1169  0.3570       0.1889  0.3661
4                        0.0303  0.8137          NaN     NaN

In [6]:
# Sensitivity check: does reusing 05's primary K = 4 instead of diesel's own K = 3
# change the result? If diesel's asymmetry were an artifact of picking a shorter lag
# count, refitting at K = 4 should look different.
X_diesel_k4 = build_design_matrix(retail_diesel, crude_wti, K=4, mode="weekly")
res_diesel_k4 = fit_distributed_lag(X_diesel_k4)

diesel_result_k4 = pd.DataFrame(
    [test_asymmetry(res_diesel_k4, K=4, horizon=h) for h in (0, 1, 4)]
).set_index("horizon")[["estimate", "p_value"]]

pd.concat({"diesel (K=3)": diesel_result, "diesel (K=4)": diesel_result_k4}, axis=1).round(4)

build_design_matrix: dropped 5 of 866 rows to NaN (differencing + 4 lag(s)); 861 rows remain


diesel (K=3)         diesel (K=4)        
            estimate p_value     estimate p_value
horizon                                          
0             0.6889  0.0022       0.6941  0.0027
1             0.5002  0.0172       0.5031  0.0198
3             0.1889  0.3661          NaN     NaN
4                NaN     NaN       0.1797  0.4156

The same-week and week-1 estimates barely move between K = 3 and K = 4 (0.69 -> 0.69 at h = 0, 0.50 -> 0.50 at h = 1), so diesel's result is not an artifact of the shorter lag count; K = 3 is used going forward because it is what diesel's own cross-correlation supports, not because it gives a more favorable answer.

## 3. Result

Brent is settled in §1: switching the crude benchmark doesn't change the conclusion, so the
robustness check `01` named as a planned role for Brent is complete.

**Diesel shows a similar asymmetric pattern to gasoline.** At its own K = 3, the diesel same-week gap is 0.69 (p = 0.002), and the week-1 gap is 0.50 (p = 0.017). Both are statistically significant and are numerically larger than the corresponding gasoline estimates of 0.43 and 0.23. Diesel is refined from the same crude but sold through a separate
retail market from gasoline, so this points to the asymmetric pass-through pattern being broader
than gasoline specifically, not confined to it. It does not change `05`'s gasoline result, which
stands on its own evidence. Where in diesel's own supply chain this asymmetry originates
(crude -> wholesale diesel vs. wholesale -> retail diesel, the same split `05` §7 ran for gasoline)
was not tested here.